In [1]:
import pandas as pd
import numpy as np
from BK_Tree import BKTree
import re
import string
from dead_processing_utils import *
from file_processing_utils import *

### Template Converter

The purpose of this notebook is to use the output files of the other notebookes to convert the doxy output to a ucbg upload template

### First: upload authority terms

taxon to add:

In [2]:
taxon_template = strip('ucbg_templates/ucbg_4-0-0_taxon-template.csv')
secondary_taxon_template = taxon_template.copy()

In [3]:
def get_rank(name):
    name = str(name)
    if len(name.split()) == 1:
        return 'genus'
    elif len(name.split()) == 2:
        return 'species'
    elif len(name.split()) == 3 and '×' in name.split():
        return 'species'
    elif 'subsp.' in name.split():
        return 'subspecies'
    elif 'var.' in name.split():
        return 'variety'
    elif 'f.' in name.split():
        return 'form'
    elif 'x' in name.split():
        return 'hybrid'
        


In [ ]:
checked_names_to_add = pd.read_csv('output/to_add_final.csv')
checked_names_to_add = checked_names_to_add.iloc[41:]

# status = (checked_names_to_add['checked_synonym'] == checked_names_to_add['accepted_name']).replace({True: 'accepted', False: 'synonym'}).values[0]
rank = checked_names_to_add['checked_synonym'].apply(
    lambda x: get_rank(x)
)

taxon_template['termDisplayName'] = checked_names_to_add['checked_synonym']
taxon_template['termType'] = 'Verified Taxonomic Name'
taxon_template['termStatus'] = 'accepted'
taxon_template['taxonomicStatus'] = 'valid'
taxon_template['termSource'] = 'UC Botanical Garden'
taxon_template['taxonRank'] = rank
taxon_template['accessRestrictions'] = 'Dead'
taxon_template['family'] = checked_names_to_add['family'].str.upper()


In [5]:
taxon_template.dropna(subset=['termDisplayName']).to_csv('output/taxon_test_upload.csv', index=False)

In [6]:
taxon_template['termDisplayName'].isna().sum()

np.int64(125)

### Next: relate authority terms to higher taxon i.e. Quercus agrifolia --> Quercus

In [7]:
authority_hierarchy = strip('ucbg_templates/ucbg_4-0-0_authorityhierarchy-template.csv')
secondary_authority_hierarchy = authority_hierarchy.copy()

In [8]:
def broader_term(name):
    name = str(name)
    if len(name.split()) == 1:
        try:
            family = taxon_template.loc[taxon_template['termDisplayName'] == name]['family'].values[0]
            return family.upper()
        except IndexError:
            return None
    elif len(name.split()) == 3 and '×' in name.split():
        return name.split()[0]
    elif len(name.split()) == 2:
        return name.split()[0]
    else:
        return ' '.join(name.split()[0:2])

In [9]:
broader_term('Maidenia')

'HYDROCHARITACEAE'

In [10]:
authority_hierarchy['narrower_term'] = taxon_template['termDisplayName'].reset_index(drop=True)
authority_hierarchy['broader_term'] = authority_hierarchy['narrower_term'].apply(broader_term)
authority_hierarchy['term_type'] = 'taxonomyauthority'
authority_hierarchy['term_subtype'] = 'taxon'

authority_hierarchy = authority_hierarchy.dropna()

authority_hierarchy.to_csv('output/authorityhierarchy_test_upload.csv', index=False)

### Generate new upload for missing authority terms

In [11]:
warning_report = pd.read_csv('output/big_authorityhierarchy_test-20251114-1906_processing_report.csv')

warning_report = warning_report.dropna(subset=['ERR: unsuccessful_csid_lookup_for_term'])
len(warning_report)

to_upload = warning_report['ERR: unsuccessful_csid_lookup_for_term']
to_upload = pd.Series(to_upload).str.split().str[1:].str.join(' ')
original_name = warning_report['narrower_term']
to_upload_df = pd.DataFrame({'to_upload': to_upload, 'original_name': original_name})
to_upload_df = pd.merge(to_upload_df, checked_names_to_add[['checked_synonym', 'family']], left_on='original_name', right_on='checked_synonym', how='left')
to_upload_df = to_upload_df[['to_upload', 'family']]

In [ ]:
rank = to_upload_df['to_upload'].apply(
    lambda x: get_rank(x)
)

secondary_taxon_template['termDisplayName'] = to_upload_df['to_upload'].reset_index(drop=True)
secondary_taxon_template['termType'] = 'Verified Taxonomic Name'
secondary_taxon_template['termStatus'] = 'accepted'
secondary_taxon_template['taxonomicStatus'] = 'valid'
secondary_taxon_template['termSource'] = 'UC Botanical Garden'
secondary_taxon_template['taxonRank'] = rank
secondary_taxon_template['accessRestrictions'] = 'Dead'
secondary_taxon_template['family'] = to_upload_df['family'].str.upper()

secondary_taxon_template = secondary_taxon_template.drop_duplicates()

secondary_taxon_template.to_csv('output/taxon_test_upload_ROUND2.csv', index=False)

In [13]:
secondary_authority_hierarchy['narrower_term'] = to_upload_df['to_upload'].reset_index(drop=True)
secondary_authority_hierarchy['broader_term'] = secondary_authority_hierarchy['narrower_term'].apply(broader_term)
secondary_authority_hierarchy['term_type'] = 'taxonomyauthority'
secondary_authority_hierarchy['term_subtype'] = 'taxon'

secondary_authority_hierarchy = secondary_authority_hierarchy.dropna()
secondary_authority_hierarchy.to_csv('output/authorityhierarchy_test_upload_ROUND2.csv', index=False)